# Module 6.2: 分布式训练 (Distributed Training)

## 1. 本章概览

### 📚 学习目标

1. **并行策略**：理解数据并行、模型并行、流水线并行
2. **分布式框架**：掌握 DDP、FSDP、DeepSpeed
3. **内存优化**：学习 ZeRO 优化器和梯度检查点
4. **实践应用**：在多 GPU 环境中训练大模型

### 🎯 核心问题

- 为什么需要分布式训练？
- 如何选择合适的并行策略？
- 如何最大化 GPU 利用率？


### 🏢 业务场景

本章技术将应用于 `电商客服智能助理` 场景：
- 大促前如何加速模型训练迭代？→ 数据并行与多卡扩展策略
- 多机训练时节点掉线如何恢复？→ 分布式容错与 checkpoint 机制

### 🔑 核心概念速览

**AllReduce** 是分布式计算中的一种集合通信操作，将所有节点上的数据进行归约（如求和）并将结果广播回每个节点。在本章场景中，它用于数据并行训练时同步各 GPU 上的梯度。

**Ring-AllReduce** 是 AllReduce 操作的一种高效实现算法，将节点组织成环形拓扑，通过分块传递使通信量与节点数无关。在本章场景中，它是 DDP 梯度同步的默认通信策略。

**ZeRO (Zero Redundancy Optimizer)** 是微软 DeepSpeed 提出的显存优化策略，通过将优化器状态、梯度和参数分片到多个 GPU 上消除冗余存储。在本章场景中，它用于突破单 GPU 显存限制，实现超大模型的分布式训练。

**Megatron-LM** 是 NVIDIA 开发的大规模语言模型训练框架，实现了高效的张量并行和流水线并行策略。在本章场景中，它展示了如何在 Transformer 层内进行矩阵切分以实现模型并行。

### ⏱️ 预计学习时间：5-6小时

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from accelerate import Accelerator
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)
print("✓ Libraries imported")

## 2. 分布式训练基础

### 2.1 为什么需要分布式训练？

**三大挑战**：

1. **模型太大**：GPT-3 (175B 参数) 需要 ~700GB 内存
   - 单个 A100 (80GB) 无法容纳
   
2. **数据太多**：训练数据集可能有 TB 级别
   - 单 GPU 训练时间过长
   
3. **训练太慢**：大模型训练可能需要数月
   - 需要加速训练过程

### 2.2 并行策略分类

| 策略 | 模型分布 | 数据分布 | 适用场景 |
|------|----------|----------|----------|
| **数据并行** | 每个 GPU 完整副本 | 分割 | 模型能放入单 GPU |
| **模型并行** | 分割到多个 GPU | 完整 | 模型太大 |
| **流水线并行** | 按层分割 | 分割 | 深层模型 |
| **混合并行** | 组合策略 | 分割 | 超大模型 |

### 2.3 内存分析

训练一个模型需要的内存：

$$\text{Memory} = \underbrace{2\Phi}_{\text{参数(FP16)}} + \underbrace{2\Phi}_{\text{梯度}} + \underbrace{12\Phi}_{\text{优化器状态}} + \underbrace{\text{Activations}}_{\text{激活值}}$$

对于 Adam 优化器：
- 参数：2 bytes/param (FP16)
- 梯度：2 bytes/param
- 优化器状态：12 bytes/param (FP32 参数 + 2个动量)

**总计**：~16 bytes/param + activations

In [ ]:
# 🔬 Micro Practice: 内存需求计算

def calculate_memory_requirements(num_params_billions, batch_size=1, seq_len=2048, hidden_size=4096):
    """
    计算训练所需内存
    
    Args:
        num_params_billions: 参数量（十亿）
        batch_size: 批次大小
        seq_len: 序列长度
        hidden_size: 隐藏层维度
    """
    num_params = num_params_billions * 1e9
    
    # 模型权重 (FP16)
    model_memory = num_params * 2 / 1e9  # GB
    
    # 梯度 (FP16)
    gradient_memory = num_params * 2 / 1e9
    
    # 优化器状态 (Adam: FP32 params + 2 momentum)
    optimizer_memory = num_params * 12 / 1e9
    
    # 激活值（粗略估计）
    activation_memory = batch_size * seq_len * hidden_size * 4 / 1e9
    
    total_memory = model_memory + gradient_memory + optimizer_memory + activation_memory
    
    print(f"模型参数量: {num_params_billions}B")
    print(f"\n内存分解:")
    print(f"  模型权重:     {model_memory:>8.2f} GB")
    print(f"  梯度:         {gradient_memory:>8.2f} GB")
    print(f"  优化器状态:   {optimizer_memory:>8.2f} GB")
    print(f"  激活值:       {activation_memory:>8.2f} GB")
    print(f"  {'='*30}")
    print(f"  总计:         {total_memory:>8.2f} GB")
    print(f"\n需要 GPU 数量 (A100 80GB): {int(np.ceil(total_memory / 80))}")
    
    return total_memory

# 测试不同规模的模型
print("=" * 50)
print("GPT-2 (1.5B 参数)")
print("=" * 50)
calculate_memory_requirements(1.5, batch_size=8)

print("\n" + "=" * 50)
print("GPT-3 (175B 参数)")
print("=" * 50)
calculate_memory_requirements(175, batch_size=1)

print("\n✓ 内存分析完成！")

## 3. 扩展挑战

### 3.1 通信开销

分布式训练的主要瓶颈：

**数据并行**：
- 需要同步梯度：$O(\Phi)$ 通信量
- AllReduce 操作

**模型并行**：
- 需要传递激活值：$O(\text{batch\_size} \times \text{hidden\_size})$
- 频繁的跨 GPU 通信

### 3.2 理想 vs 实际加速比

**理想情况**：使用 N 个 GPU，速度提升 N 倍

**实际情况**：
$$\text{Speedup} = \frac{N}{1 + \alpha(N-1)}$$

其中 $\alpha$ 是通信开销比例。

### 3.3 可视化扩展效率

In [ ]:
# 🔬 Micro Practice: 扩展效率可视化

def plot_scaling_efficiency():
    num_gpus = np.arange(1, 65)
    
    # 不同通信开销下的加速比
    alphas = [0.0, 0.05, 0.1, 0.2]
    
    plt.figure(figsize=(12, 5))
    
    # 加速比
    plt.subplot(1, 2, 1)
    for alpha in alphas:
        speedup = num_gpus / (1 + alpha * (num_gpus - 1))
        label = f"α={alpha:.2f}" if alpha > 0 else "理想情况"
        plt.plot(num_gpus, speedup, label=label, linewidth=2)
    
    plt.plot(num_gpus, num_gpus, 'k--', alpha=0.3, label='线性加速')
    plt.xlabel('GPU 数量', fontsize=12)
    plt.ylabel('加速比', fontsize=12)
    plt.title('分布式训练加速比', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 效率
    plt.subplot(1, 2, 2)
    for alpha in alphas:
        speedup = num_gpus / (1 + alpha * (num_gpus - 1))
        efficiency = speedup / num_gpus * 100
        label = f"α={alpha:.2f}" if alpha > 0 else "理想情况"
        plt.plot(num_gpus, efficiency, label=label, linewidth=2)
    
    plt.xlabel('GPU 数量', fontsize=12)
    plt.ylabel('效率 (%)', fontsize=12)
    plt.title('扩展效率', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim([0, 105])
    
    plt.tight_layout()
    plt.show()
    
    # 打印关键观察
    print("关键观察:")
    print("1. 通信开销 α 越大，扩展效率下降越快")
    print("2. GPU 数量增加，边际收益递减")
    print("3. 需要优化通信以提高扩展效率")

plot_scaling_efficiency()
print("\n✓ 扩展分析完成！")

## 4. 数据并行 (Data Parallelism)

### 4.1 数据并行原理

**核心思想**：每个 GPU 持有完整模型副本，处理不同的数据批次

**训练流程**：

```
1. Forward Pass (并行):
   GPU 0: batch_0 → loss_0
   GPU 1: batch_1 → loss_1
   GPU 2: batch_2 → loss_2
   GPU 3: batch_3 → loss_3

2. Backward Pass (并行):
   GPU 0: ∇L_0
   GPU 1: ∇L_1
   GPU 2: ∇L_2
   GPU 3: ∇L_3

3. Gradient Synchronization (AllReduce):
   ∇L_avg = (∇L_0 + ∇L_1 + ∇L_2 + ∇L_3) / 4

4. Parameter Update (同步):
   θ ← θ - η * ∇L_avg
```

### 4.2 DataParallel vs DistributedDataParallel

| 特性 | DataParallel (DP) | DistributedDataParallel (DDP) |
|------|-------------------|-------------------------------|
| 进程模型 | 单进程多线程 | 多进程 |
| 通信方式 | 主 GPU 收集梯度 | AllReduce |
| 负载均衡 | 主 GPU 负载重 | 均衡 |
| 性能 | 较慢 | 快 |
| 推荐使用 | ❌ 不推荐 | ✅ 推荐 |

### 4.3 AllReduce 操作

**Ring-AllReduce 算法**：

$$\text{Time} = 2(N-1) \times \frac{M}{N \times B}$$

其中：
- $N$: GPU 数量
- $M$: 数据大小
- $B$: 带宽

**优势**：通信时间与 GPU 数量无关（理想情况）

In [ ]:
# 🔬 Micro Practice: DataParallel 示例

class SimpleModel(nn.Module):
    def __init__(self, input_size=1024, hidden_size=2048, output_size=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 检查可用 GPU
num_gpus = torch.cuda.device_count()
print(f"可用 GPU 数量: {num_gpus}")

if num_gpus > 1:
    print("\n使用 DataParallel:")
    model = SimpleModel()
    model = nn.DataParallel(model)
    model = model.cuda()
    
    # 测试前向传播
    batch_size = 32
    x = torch.randn(batch_size, 1024).cuda()
    output = model(x)
    
    print(f"输入形状: {x.shape}")
    print(f"输出形状: {output.shape}")
    print(f"模型分布在 GPU: {list(range(num_gpus))}")
else:
    print("\n单 GPU 模式（需要多个 GPU 才能演示数据并行）")
    model = SimpleModel()
    if torch.cuda.is_available():
        model = model.cuda()
        x = torch.randn(32, 1024).cuda()
    else:
        x = torch.randn(32, 1024)
    
    output = model(x)
    print(f"输入形状: {x.shape}")
    print(f"输出形状: {output.shape}")

print("\n✓ DataParallel 示例完成！")
print("\n注意: DataParallel 有性能瓶颈，实际应用中应使用 DDP")

## 5. DistributedDataParallel (DDP)

### 5.1 DDP 架构

**多进程设计**：
- 每个 GPU 对应一个独立进程
- 使用 NCCL 后端进行高效通信
- 梯度在反向传播时自动同步

**关键组件**：

```python
# 1. 初始化进程组
dist.init_process_group(backend='nccl')

# 2. 设置设备
torch.cuda.set_device(local_rank)

# 3. 包装模型
model = DDP(model, device_ids=[local_rank])

# 4. 使用 DistributedSampler
sampler = DistributedSampler(dataset)
dataloader = DataLoader(dataset, sampler=sampler)
```

### 5.2 梯度同步机制

**Gradient Bucketing**：
- 将梯度分组到 buckets (~25MB)
- 每个 bucket 完成后立即 AllReduce
- 重叠计算和通信

**通信优化**：
$$\text{Overlap Ratio} = \frac{\text{Compute Time}}{\text{Communication Time}}$$

目标：Overlap Ratio > 1（计算掩盖通信）

### 5.3 DDP 最佳实践

1. **使用 NCCL 后端**（GPU 通信最快）
2. **设置 find_unused_parameters=False**（除非必要）
3. **使用梯度累积**减少通信频率
4. **Pin memory** 加速数据传输

In [ ]:
# 🔬 Micro Practice: DDP 训练脚本框架

ddp_script = '''
# train_ddp.py - DDP 训练脚本示例

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler
import os

def setup(rank, world_size):
    """初始化分布式环境"""
    os.environ[\'MASTER_ADDR\'] = \'localhost\'
    os.environ[\'MASTER_PORT\'] = \'12355\'
    
    # 初始化进程组
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup():
    """清理分布式环境"""
    dist.destroy_process_group()

def train(rank, world_size):
    print(f"Running DDP on rank {rank}.")
    setup(rank, world_size)
    
    # 创建模型并移到 GPU
    model = YourModel().to(rank)
    ddp_model = DDP(model, device_ids=[rank])
    
    # 创建数据加载器
    dataset = YourDataset()
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    dataloader = DataLoader(dataset, batch_size=32, sampler=sampler, 
                           pin_memory=True, num_workers=4)
    
    # 优化器
    optimizer = torch.optim.AdamW(ddp_model.parameters(), lr=1e-4)
    
    # 训练循环
    for epoch in range(num_epochs):
        sampler.set_epoch(epoch)  # 重要：确保每个 epoch 数据打乱不同
        
        for batch in dataloader:
            inputs, labels = batch
            inputs = inputs.to(rank)
            labels = labels.to(rank)
            
            # 前向传播
            outputs = ddp_model(inputs)
            loss = criterion(outputs, labels)
            
            # 反向传播（梯度自动同步）
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            if rank == 0:  # 只在主进程打印
                print(f"Epoch {epoch}, Loss: {loss.item():.4f}")
    
    cleanup()

if __name__ == "__main__":
    world_size = torch.cuda.device_count()
    torch.multiprocessing.spawn(train, args=(world_size,), nprocs=world_size)
'''

print("DDP 训练脚本框架:")
print("=" * 60)
print(ddp_script)
print("=" * 60)

print("\n启动命令:")
print("  单机多卡: python train_ddp.py")
print("  多机多卡: torchrun --nproc_per_node=8 --nnodes=2 train_ddp.py")

print("\n✓ DDP 示例完成！")

## 6. 模型并行 (Model Parallelism)

### 6.1 模型并行原理

**核心思想**：将模型的不同部分放在不同的 GPU 上

**两种主要方式**：

1. **张量并行 (Tensor Parallelism)**：
   - 将单个层的权重矩阵切分到多个 GPU
   - 适合超大层（如 FFN）

2. **流水线并行 (Pipeline Parallelism)**：
   - 将不同的层放在不同的 GPU
   - 适合深层模型

### 6.2 张量并行详解

**列切分 (Column Parallel)**：

$$Y = XW \rightarrow Y = [XW_1, XW_2]$$

```
GPU 0: X @ W[:, :d/2] → Y[:, :d/2]
GPU 1: X @ W[:, d/2:] → Y[:, d/2:]
结果: Concatenate([Y_0, Y_1])
```

**行切分 (Row Parallel)**：

$$Y = XW \rightarrow Y = X_1W_1 + X_2W_2$$

```
GPU 0: X[:, :d/2] @ W[:d/2, :] → Y_0
GPU 1: X[:, d/2:] @ W[d/2:, :] → Y_1
结果: AllReduce(Y_0 + Y_1)
```

### 6.3 Megatron-LM 策略

**Transformer 层的切分**：

```
Self-Attention:
  Q, K, V: 列切分（输出拼接）
  Output: 行切分（AllReduce）

FFN:
  FC1: 列切分
  FC2: 行切分
```

**通信量**：每层 2 次 AllReduce

In [ ]:
# 🔬 Micro Practice: 简单的张量并行实现

class ColumnParallelLinear(nn.Module):
    """列并行线性层"""
    def __init__(self, in_features, out_features, num_partitions=2):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.num_partitions = num_partitions
        
        # 每个分区的输出维度
        self.out_features_per_partition = out_features // num_partitions
        
        # 创建分区权重
        self.weight = nn.Parameter(
            torch.randn(in_features, self.out_features_per_partition)
        )
        self.bias = nn.Parameter(torch.zeros(self.out_features_per_partition))
    
    def forward(self, x):
        # 本地计算
        output = torch.matmul(x, self.weight) + self.bias
        return output

class RowParallelLinear(nn.Module):
    """行并行线性层"""
    def __init__(self, in_features, out_features, num_partitions=2):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.num_partitions = num_partitions
        
        # 每个分区的输入维度
        self.in_features_per_partition = in_features // num_partitions
        
        # 创建分区权重
        self.weight = nn.Parameter(
            torch.randn(self.in_features_per_partition, out_features)
        )
        self.bias = nn.Parameter(torch.zeros(out_features))
    
    def forward(self, x):
        # 本地计算
        output = torch.matmul(x, self.weight)
        # 实际应用中这里需要 AllReduce
        output = output + self.bias
        return output

# 演示
print("张量并行示例:")
print("=" * 60)

batch_size, seq_len = 4, 128
d_model = 512
d_ff = 2048

# 模拟 2 个 GPU
num_gpus = 2

# 列并行层（如 FFN 的第一层）
col_parallel = ColumnParallelLinear(d_model, d_ff, num_partitions=num_gpus)
x = torch.randn(batch_size, seq_len, d_model)
output_col = col_parallel(x)

print(f"输入形状: {x.shape}")
print(f"列并行输出形状: {output_col.shape}")
print(f"  (每个 GPU 输出 {d_ff // num_gpus} 维，需要拼接)")

# 行并行层（如 FFN 的第二层）
row_parallel = RowParallelLinear(d_ff, d_model, num_partitions=num_gpus)
x_split = torch.randn(batch_size, seq_len, d_ff // num_gpus)
output_row = row_parallel(x_split)

print(f"\n行并行输入形状: {x_split.shape}")
print(f"行并行输出形状: {output_row.shape}")
print(f"  (每个 GPU 输出完整维度，需要 AllReduce 求和)")

print("\n通信模式:")
print("  列并行: 无通信 → 拼接 (带宽需求低)")
print("  行并行: AllReduce (带宽需求高)")

print("\n✓ 张量并行示例完成！")

## 7. 流水线并行 (Pipeline Parallelism)

### 7.1 流水线并行原理

**核心思想**：将模型按层切分成多个阶段（stage），每个阶段在不同 GPU

**朴素流水线问题**：

```
时间 →
GPU 0: [■■■■] .... .... ....  (计算 stage 0)
GPU 1: .... [■■■■] .... ....  (等待 → 计算 stage 1)
GPU 2: .... .... [■■■■] ....  (等待 → 等待 → 计算 stage 2)
GPU 3: .... .... .... [■■■■]  (等待 → 等待 → 等待 → 计算 stage 3)

问题: 大量 GPU 空闲时间（气泡）
```

### 7.2 微批次 (Micro-batching)

**解决方案**：将批次切分成多个微批次

```
时间 →
GPU 0: [■][■][■][■] .... .... ....  (处理 4 个微批次)
GPU 1: .[■][■][■][■] .... ....      (流水线填充)
GPU 2: ..[■][■][■][■] ....           (减少气泡)
GPU 3: ...[■][■][■][■]                (提高利用率)

改进: GPU 利用率显著提升
```

### 7.3 GPipe 调度

**前向传播**：
$$F_{i,j}: \text{GPU } i \text{ 处理微批次 } j \text{ 的前向传播}$$

**反向传播**：
$$B_{i,j}: \text{GPU } i \text{ 处理微批次 } j \text{ 的反向传播}$$

**调度顺序**：
```
阶段 1: 所有微批次的前向传播
阶段 2: 所有微批次的反向传播
阶段 3: 梯度更新
```

### 7.4 气泡时间分析

**气泡比例**：
$$\text{Bubble Ratio} = \frac{p-1}{m+p-1}$$

其中：
- $p$: 流水线阶段数（GPU 数）
- $m$: 微批次数量

**优化目标**：增加 $m$ 减少气泡，但会增加内存使用

In [ ]:
# 🔬 Micro Practice: 流水线并行可视化

def visualize_pipeline(num_stages=4, num_microbatches=8):
    """
    可视化流水线并行调度
    """
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # 朴素流水线（无微批次）
    ax1 = axes[0]
    for stage in range(num_stages):
        # 前向传播
        ax1.barh(stage, 1, left=stage, color='skyblue', edgecolor='black')
        # 反向传播
        ax1.barh(stage, 1, left=2*num_stages-stage-1, color='salmon', edgecolor='black')
    
    ax1.set_yticks(range(num_stages))
    ax1.set_yticklabels([f'GPU {i}' for i in range(num_stages)])
    ax1.set_xlabel('时间步', fontsize=12)
    ax1.set_title('朴素流水线（无微批次）- 大量气泡时间', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')
    ax1.set_xlim(0, 2*num_stages)
    
    # 添加图例
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='skyblue', edgecolor='black', label='前向传播'),
        Patch(facecolor='salmon', edgecolor='black', label='反向传播')
    ]
    ax1.legend(handles=legend_elements, loc='upper right')
    
    # GPipe 流水线（微批次）
    ax2 = axes[1]
    
    # 前向传播阶段
    for mb in range(num_microbatches):
        for stage in range(num_stages):
            time_start = mb + stage
            ax2.barh(stage, 0.9, left=time_start, color='skyblue', edgecolor='black', linewidth=0.5)
    
    # 反向传播阶段
    for mb in range(num_microbatches):
        for stage in range(num_stages):
            time_start = num_microbatches + num_stages - 1 + mb + (num_stages - stage - 1)
            ax2.barh(stage, 0.9, left=time_start, color='salmon', edgecolor='black', linewidth=0.5)
    
    ax2.set_yticks(range(num_stages))
    ax2.set_yticklabels([f'GPU {i}' for i in range(num_stages)])
    ax2.set_xlabel('时间步', fontsize=12)
    ax2.set_title(f'GPipe 流水线（{num_microbatches} 个微批次）- 气泡时间减少', 
                 fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')
    ax2.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()
    
    # 计算气泡比例
    bubble_ratio = (num_stages - 1) / (num_microbatches + num_stages - 1)
    print(f"\n气泡时间分析:")
    print(f"  流水线阶段数: {num_stages}")
    print(f"  微批次数量: {num_microbatches}")
    print(f"  气泡比例: {bubble_ratio:.2%}")
    print(f"  有效利用率: {(1-bubble_ratio):.2%}")

visualize_pipeline(num_stages=4, num_microbatches=8)

print("\n关键观察:")
print("1. 微批次数量越多，气泡比例越小")
print("2. 但微批次过多会增加内存占用（需要存储中间激活值）")
print("3. 需要在利用率和内存之间权衡")

print("\n✓ 流水线并行可视化完成！")

In [ ]:
# 🔬 Micro Practice: 简单的流水线并行实现

class PipelineStage(nn.Module):
    """流水线的一个阶段"""
    def __init__(self, layers):
        super().__init__()
        self.layers = nn.ModuleList(layers)
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

class SimplePipelineParallel:
    """简单的流水线并行实现"""
    def __init__(self, model_layers, num_stages=4):
        self.num_stages = num_stages
        layers_per_stage = len(model_layers) // num_stages
        
        # 将层分配到不同阶段
        self.stages = []
        for i in range(num_stages):
            start_idx = i * layers_per_stage
            end_idx = start_idx + layers_per_stage if i < num_stages - 1 else len(model_layers)
            stage_layers = model_layers[start_idx:end_idx]
            self.stages.append(PipelineStage(stage_layers))
    
    def forward_microbatch(self, microbatch):
        """处理一个微批次"""
        x = microbatch
        for stage in self.stages:
            x = stage(x)
        return x
    
    def forward(self, batch, num_microbatches=4):
        """使用微批次处理完整批次"""
        microbatch_size = batch.size(0) // num_microbatches
        outputs = []
        
        for i in range(num_microbatches):
            start_idx = i * microbatch_size
            end_idx = start_idx + microbatch_size
            microbatch = batch[start_idx:end_idx]
            
            output = self.forward_microbatch(microbatch)
            outputs.append(output)
        
        return torch.cat(outputs, dim=0)

# 演示
print("流水线并行示例:")
print("=" * 60)

# 创建一个简单的模型（12 层）
layers = [nn.Linear(512, 512) for _ in range(12)]

# 创建流水线（4 个阶段，每个阶段 3 层）
pipeline = SimplePipelineParallel(layers, num_stages=4)

print(f"模型总层数: 12")
print(f"流水线阶段数: 4")
print(f"每个阶段层数: 3")

# 测试
batch_size = 32
num_microbatches = 8
x = torch.randn(batch_size, 512)

print(f"\n批次大小: {batch_size}")
print(f"微批次数量: {num_microbatches}")
print(f"每个微批次大小: {batch_size // num_microbatches}")

output = pipeline.forward(x, num_microbatches=num_microbatches)
print(f"\n输出形状: {output.shape}")

# 计算气泡比例
bubble_ratio = (4 - 1) / (num_microbatches + 4 - 1)
print(f"\n气泡比例: {bubble_ratio:.2%}")
print(f"有效利用率: {(1-bubble_ratio):.2%}")

print("\n✓ 流水线并行示例完成！")

## 8. ZeRO 优化器 (Zero Redundancy Optimizer)

### 8.1 ZeRO 动机

**问题**：数据并行中，每个 GPU 都存储完整的：
- 模型参数
- 梯度
- 优化器状态

这造成了 **内存冗余**！

### 8.2 ZeRO 三个阶段

**ZeRO-1: 优化器状态分片**
```
传统 DDP (N GPUs):
  每个 GPU: 参数 + 梯度 + 优化器状态
  内存: N × (2Φ + 2Φ + 12Φ) = 16NΦ

ZeRO-1:
  每个 GPU: 参数 + 梯度 + 优化器状态/N
  内存: N × (2Φ + 2Φ + 12Φ/N) = 4NΦ + 12Φ
  节省: ~4× (当 N 较大时)
```

**ZeRO-2: + 梯度分片**
```
每个 GPU: 参数 + 梯度/N + 优化器状态/N
内存: N × (2Φ + 2Φ/N + 12Φ/N) = 2NΦ + 14Φ
节省: ~8× (当 N 较大时)
```

**ZeRO-3: + 参数分片**
```
每个 GPU: 参数/N + 梯度/N + 优化器状态/N
内存: N × (2Φ/N + 2Φ/N + 12Φ/N) = 16Φ
节省: N× (线性扩展！)
```

### 8.3 ZeRO-3 工作流程

```
前向传播:
  1. AllGather 当前层的参数
  2. 计算前向传播
  3. 丢弃参数（只保留本 GPU 的分片）

反向传播:
  1. AllGather 当前层的参数
  2. 计算反向传播
  3. ReduceScatter 梯度
  4. 丢弃参数

参数更新:
  每个 GPU 只更新自己的参数分片
```

### 8.4 ZeRO-Offload

进一步优化：将优化器状态和梯度卸载到 CPU/NVMe

$$\text{GPU Memory} \approx 2\Phi \text{ (仅参数)}$$

In [ ]:
# 🔬 Micro Practice: ZeRO 内存节省对比

def compare_zero_stages(num_params_billions, num_gpus_list=[1, 4, 8, 16]):
    """
    比较不同 ZeRO 阶段的内存使用
    """
    num_params = num_params_billions * 1e9
    
    print(f"模型参数量: {num_params_billions}B")
    print("=" * 80)
    
    results = []
    
    for num_gpus in num_gpus_list:
        # 标准 DDP
        ddp_memory = 16 * num_params / 1e9  # 每个 GPU
        
        # ZeRO-1: 优化器状态分片
        zero1_memory = (4 * num_params + 12 * num_params / num_gpus) / 1e9
        
        # ZeRO-2: + 梯度分片
        zero2_memory = (2 * num_params + 14 * num_params / num_gpus) / 1e9
        
        # ZeRO-3: + 参数分片
        zero3_memory = 16 * num_params / num_gpus / 1e9
        
        results.append({
            'gpus': num_gpus,
            'ddp': ddp_memory,
            'zero1': zero1_memory,
            'zero2': zero2_memory,
            'zero3': zero3_memory
        })
        
        print(f"\n{num_gpus} GPUs:")
        print(f"  DDP:    {ddp_memory:>8.2f} GB/GPU")
        print(f"  ZeRO-1: {zero1_memory:>8.2f} GB/GPU (节省 {ddp_memory/zero1_memory:.1f}×)")
        print(f"  ZeRO-2: {zero2_memory:>8.2f} GB/GPU (节省 {ddp_memory/zero2_memory:.1f}×)")
        print(f"  ZeRO-3: {zero3_memory:>8.2f} GB/GPU (节省 {ddp_memory/zero3_memory:.1f}×)")
    
    # 可视化
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 内存使用对比
    x = np.arange(len(num_gpus_list))
    width = 0.2
    
    ax1.bar(x - 1.5*width, [r['ddp'] for r in results], width, label='DDP', color='#e74c3c')
    ax1.bar(x - 0.5*width, [r['zero1'] for r in results], width, label='ZeRO-1', color='#f39c12')
    ax1.bar(x + 0.5*width, [r['zero2'] for r in results], width, label='ZeRO-2', color='#3498db')
    ax1.bar(x + 1.5*width, [r['zero3'] for r in results], width, label='ZeRO-3', color='#2ecc71')
    
    ax1.set_xlabel('GPU 数量', fontsize=12)
    ax1.set_ylabel('内存使用 (GB/GPU)', fontsize=12)
    ax1.set_title('ZeRO 内存使用对比', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels([f"{n}" for n in num_gpus_list])
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # 节省倍数
    for i, stage in enumerate(['ZeRO-1', 'ZeRO-2', 'ZeRO-3']):
        key = stage.lower().replace('-', '')
        savings = [results[j]['ddp'] / results[j][key] for j in range(len(results))]
        ax2.plot(num_gpus_list, savings, marker='o', linewidth=2, markersize=8, label=stage)
    
    ax2.set_xlabel('GPU 数量', fontsize=12)
    ax2.set_ylabel('内存节省倍数', fontsize=12)
    ax2.set_title('ZeRO 内存节省效果', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

compare_zero_stages(num_params_billions=7, num_gpus_list=[1, 4, 8, 16])

print("\n关键观察:")
print("1. ZeRO-3 实现了线性内存扩展")
print("2. GPU 数量越多，ZeRO-3 的优势越明显")
print("3. ZeRO-3 可以训练单 GPU 无法容纳的模型")

print("\n✓ ZeRO 对比完成！")

## 9. DeepSpeed 框架

### 9.1 DeepSpeed 简介

**DeepSpeed** 是微软开发的分布式训练框架，实现了 ZeRO 优化器。

**核心特性**：
- ZeRO-1/2/3 优化
- ZeRO-Offload (CPU/NVMe)
- 梯度检查点
- 混合精度训练
- 流水线并行

### 9.2 DeepSpeed 配置

```json
{
  "train_batch_size": 32,
  "gradient_accumulation_steps": 1,
  "optimizer": {
    "type": "AdamW",
    "params": {
      "lr": 1e-4,
      "betas": [0.9, 0.999],
      "eps": 1e-8
    }
  },
  "fp16": {
    "enabled": true
  },
  "zero_optimization": {
    "stage": 3,
    "offload_optimizer": {
      "device": "cpu",
      "pin_memory": true
    },
    "offload_param": {
      "device": "cpu",
      "pin_memory": true
    }
  }
}
```

### 9.3 使用 DeepSpeed

```python
import deepspeed

# 初始化
model_engine, optimizer, _, _ = deepspeed.initialize(
    model=model,
    config=ds_config
)

# 训练循环
for batch in dataloader:
    loss = model_engine(batch)
    model_engine.backward(loss)
    model_engine.step()
```

In [ ]:
# 🔬 Micro Practice: DeepSpeed 配置示例

deepspeed_configs = {
    "ZeRO-1": {
        "train_batch_size": 32,
        "zero_optimization": {
            "stage": 1
        },
        "fp16": {"enabled": True}
    },
    "ZeRO-2": {
        "train_batch_size": 32,
        "zero_optimization": {
            "stage": 2,
            "contiguous_gradients": True,
            "overlap_comm": True
        },
        "fp16": {"enabled": True}
    },
    "ZeRO-3": {
        "train_batch_size": 32,
        "zero_optimization": {
            "stage": 3,
            "stage3_prefetch_bucket_size": 5e8,
            "stage3_param_persistence_threshold": 1e6
        },
        "fp16": {"enabled": True}
    },
    "ZeRO-3-Offload": {
        "train_batch_size": 32,
        "zero_optimization": {
            "stage": 3,
            "offload_optimizer": {
                "device": "cpu",
                "pin_memory": True
            },
            "offload_param": {
                "device": "cpu",
                "pin_memory": True
            }
        },
        "fp16": {"enabled": True}
    }
}

print("DeepSpeed 配置示例:")
print("=" * 60)

for name, config in deepspeed_configs.items():
    print(f"\n{name}:")
    import json as json_module
    print(json_module.dumps(config, indent=2))

print("\n使用建议:")
print("  ZeRO-1: 模型能放入单 GPU，需要加速")
print("  ZeRO-2: 模型较大，需要更多内存节省")
print("  ZeRO-3: 模型非常大，需要跨 GPU 分片")
print("  ZeRO-3-Offload: 极大模型，利用 CPU 内存")

print("\n✓ DeepSpeed 配置示例完成！")

## 10. FSDP (Fully Sharded Data Parallel)

### 10.1 FSDP 简介

**FSDP** 是 PyTorch 原生的分布式训练方案，类似于 ZeRO-3。

**优势**：
- PyTorch 原生支持
- 更简单的 API
- 与 PyTorch 生态系统无缝集成

### 10.2 FSDP 工作原理

与 ZeRO-3 类似：
1. **参数分片**：每个 GPU 只存储部分参数
2. **AllGather**：计算时收集完整参数
3. **计算**：前向/反向传播
4. **释放**：计算后释放非本地参数
5. **ReduceScatter**：分发梯度

### 10.3 FSDP 使用

```python
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy

# 包装模型
model = FSDP(
    model,
    auto_wrap_policy=size_based_auto_wrap_policy,
    mixed_precision=mixed_precision_policy,
    device_id=torch.cuda.current_device()
)

# 正常训练
for batch in dataloader:
    loss = model(batch)
    loss.backward()
    optimizer.step()
```

### 10.4 FSDP vs DeepSpeed

| 特性 | FSDP | DeepSpeed |
|------|------|----------|
| 集成 | PyTorch 原生 | 独立库 |
| API | 简单 | 更多配置选项 |
| CPU Offload | 支持 | 支持（更成熟）|
| NVMe Offload | 不支持 | 支持 |
| 性能 | 相当 | 相当 |
| 生态 | PyTorch | 跨框架 |

In [ ]:
# 🔬 Micro Practice: FSDP 使用示例

fsdp_script = '''
# train_fsdp.py - FSDP 训练脚本示例

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import (
    size_based_auto_wrap_policy,
    enable_wrap,
    wrap,
)

def setup(rank, world_size):
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def train(rank, world_size):
    setup(rank, world_size)
    
    # 创建模型
    model = YourModel().to(rank)
    
    # 自动包装策略：大于 1e8 参数的模块会被包装
    auto_wrap_policy = functools.partial(
        size_based_auto_wrap_policy,
        min_num_params=1e8
    )
    
    # 包装为 FSDP
    model = FSDP(
        model,
        auto_wrap_policy=auto_wrap_policy,
        device_id=rank,
        # 混合精度
        mixed_precision=MixedPrecision(
            param_dtype=torch.float16,
            reduce_dtype=torch.float16,
            buffer_dtype=torch.float16,
        ),
        # CPU offload
        cpu_offload=CPUOffload(offload_params=True),
    )
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    
    # 训练循环
    for epoch in range(num_epochs):
        for batch in dataloader:
            inputs, labels = batch
            inputs = inputs.to(rank)
            labels = labels.to(rank)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    dist.destroy_process_group()

if __name__ == "__main__":
    world_size = torch.cuda.device_count()
    torch.multiprocessing.spawn(train, args=(world_size,), nprocs=world_size)
'''

print("FSDP 训练脚本示例:")
print("=" * 60)
print(fsdp_script)
print("=" * 60)

print("\nFSDP 关键特性:")
print("  1. auto_wrap_policy: 自动决定哪些模块需要分片")
print("  2. mixed_precision: 混合精度训练")
print("  3. cpu_offload: CPU 内存卸载")
print("  4. 与 PyTorch 无缝集成")

print("\n✓ FSDP 示例完成！")

## 11. Accelerate 框架

### 11.1 Accelerate 简介

**Accelerate** 是 Hugging Face 开发的统一分布式训练框架。

**核心理念**：
- 一套代码，多种后端
- 自动设备管理
- 简化分布式训练

### 11.2 支持的后端

- 单 GPU
- 多 GPU (DDP)
- FSDP
- DeepSpeed
- TPU
- Apple Silicon (MPS)

### 11.3 使用 Accelerate

```python
from accelerate import Accelerator

# 初始化
accelerator = Accelerator()

# 准备模型、优化器、数据加载器
model, optimizer, dataloader = accelerator.prepare(
    model, optimizer, dataloader
)

# 训练循环（与单 GPU 代码相同！）
for batch in dataloader:
    outputs = model(batch)
    loss = criterion(outputs, labels)
    
    accelerator.backward(loss)
    optimizer.step()
    optimizer.zero_grad()
```

### 11.4 配置文件

```bash
# 生成配置
accelerate config

# 启动训练
accelerate launch train.py
```

In [ ]:
# 🔬 Micro Practice: Accelerate 使用示例

accelerate_script = '''
# train_accelerate.py - Accelerate 训练脚本

from accelerate import Accelerator
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def main():
    # 初始化 Accelerator
    accelerator = Accelerator(
        mixed_precision="fp16",  # 混合精度
        gradient_accumulation_steps=4  # 梯度累积
    )
    
    # 创建模型、优化器、数据加载器
    model = YourModel()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    train_dataloader = DataLoader(train_dataset, batch_size=32)
    
    # 准备（自动处理设备、分布式等）
    model, optimizer, train_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader
    )
    
    # 训练循环（与单 GPU 代码几乎相同！）
    for epoch in range(num_epochs):
        model.train()
        for batch in train_dataloader:
            with accelerator.accumulate(model):
                outputs = model(batch[\'input_ids\'])
                loss = criterion(outputs, batch[\'labels\'])
                
                # 使用 accelerator.backward 而不是 loss.backward()
                accelerator.backward(loss)
                
                optimizer.step()
                optimizer.zero_grad()
        
        # 只在主进程打印
        if accelerator.is_main_process:
            print(f"Epoch {epoch} completed")
    
    # 保存模型（自动处理分布式）
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    accelerator.save(unwrapped_model.state_dict(), "model.pt")

if __name__ == "__main__":
    main()
'''

print("Accelerate 训练脚本示例:")
print("=" * 60)
print(accelerate_script)
print("=" * 60)

print("\nAccelerate 优势:")
print("  1. 一套代码适配多种后端")
print("  2. 自动设备管理")
print("  3. 简化的 API")
print("  4. 与 Transformers 库完美集成")

print("\n配置示例:")
print("  accelerate config  # 交互式配置")
print("  accelerate launch train.py  # 启动训练")
print("  accelerate launch --num_processes 8 train.py  # 指定 GPU 数")

print("\n✓ Accelerate 示例完成！")

## 12. 实践指南与决策树

### 12.1 选择并行策略

```
模型能放入单 GPU？
├─ 是 → 使用数据并行 (DDP)
│       ├─ 需要更多内存？→ ZeRO-1/2
│       └─ 性能足够？→ 标准 DDP
│
└─ 否 → 模型太大
        ├─ 层数很多？→ 流水线并行
        ├─ 单层很大？→ 张量并行
        └─ 都很大？→ 混合并行
                    ├─ ZeRO-3 / FSDP
                    └─ + CPU Offload
```

### 12.2 框架选择

| 场景 | 推荐框架 | 原因 |
|------|----------|------|
| 模型 < 10B | DDP | 简单高效 |
| 模型 10B-100B | FSDP / DeepSpeed ZeRO-3 | 内存优化 |
| 模型 > 100B | DeepSpeed ZeRO-3 + Offload | 极致优化 |
| 快速原型 | Accelerate | 统一接口 |
| 生产环境 | DeepSpeed / FSDP | 成熟稳定 |

### 12.3 性能优化检查清单

**数据加载**：
- ✅ 使用 `pin_memory=True`
- ✅ 设置合适的 `num_workers`
- ✅ 使用 `DistributedSampler`

**通信优化**：
- ✅ 使用 NCCL 后端
- ✅ 启用梯度累积减少通信
- ✅ 使用混合精度（减少通信量）

**内存优化**：
- ✅ 梯度检查点
- ✅ 激活值重计算
- ✅ CPU/NVMe Offload

**计算优化**：
- ✅ 混合精度训练
- ✅ Flash Attention
- ✅ 编译优化（torch.compile）

In [ ]:
# 🔬 Micro Practice: 不同策略的性能对比

def plot_strategy_comparison():
    """
    可视化不同并行策略的性能特征
    """
    strategies = ['DDP', 'ZeRO-1', 'ZeRO-2', 'ZeRO-3', 'FSDP', 'Pipeline']
    
    # 模拟数据（相对值）
    metrics = {
        '吞吐量': [100, 95, 90, 75, 75, 60],
        '内存效率': [20, 40, 60, 95, 95, 70],
        '易用性': [90, 85, 85, 80, 85, 50],
        '扩展性': [70, 75, 80, 95, 95, 85]
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
    
    for idx, (metric, values) in enumerate(metrics.items()):
        ax = axes[idx]
        bars = ax.bar(strategies, values, color=colors)
        
        ax.set_ylabel('相对得分', fontsize=11)
        ax.set_title(metric, fontsize=13, fontweight='bold')
        ax.set_ylim([0, 105])
        ax.grid(True, alpha=0.3, axis='y')
        
        # 添加数值标签
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height)}',
                   ha='center', va='bottom', fontsize=9)
        
        ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print("\n策略选择建议:")
    print("  DDP: 最快，但内存受限")
    print("  ZeRO-1/2: 平衡性能和内存")
    print("  ZeRO-3/FSDP: 最佳内存效率，适合大模型")
    print("  Pipeline: 适合深层模型，需要调优")

plot_strategy_comparison()
print("\n✓ 策略对比完成！")

## 13. 常见问题 (FAQ)

### Q1: DDP vs FSDP vs DeepSpeed，如何选择？

**简单决策**：
- 模型 < 10B 参数：**DDP**（最快）
- 模型 10B-100B：**FSDP** 或 **DeepSpeed ZeRO-3**（内存优化）
- 模型 > 100B：**DeepSpeed ZeRO-3 + Offload**（极致优化）
- 快速原型：**Accelerate**（统一接口）

### Q2: 如何调试分布式训练？

**常见问题**：
1. **挂起/死锁**：
   - 检查所有进程是否执行相同的集合通信
   - 使用 `NCCL_DEBUG=INFO` 查看通信日志
   
2. **OOM（内存溢出）**：
   - 减小批次大小
   - 启用梯度检查点
   - 使用 ZeRO-3 或 CPU Offload
   
3. **性能差**：
   - 检查通信/计算比例
   - 增加批次大小
   - 使用梯度累积

### Q3: 通信开销如何优化？

**优化策略**：
1. **减少通信频率**：梯度累积
2. **减少通信量**：混合精度（FP16）
3. **重叠通信和计算**：DDP 的 gradient bucketing
4. **使用高速互联**：InfiniBand > Ethernet

### Q4: 如何处理检查点（Checkpoint）？

**ZeRO-3/FSDP 特殊处理**：
```python
# 保存
if accelerator.is_main_process:
    unwrapped_model = accelerator.unwrap_model(model)
    torch.save(unwrapped_model.state_dict(), 'checkpoint.pt')

# 加载
model.load_state_dict(torch.load('checkpoint.pt'))
```

**DeepSpeed**：
```python
# 保存
model_engine.save_checkpoint('checkpoints', tag='step_1000')

# 加载
model_engine.load_checkpoint('checkpoints', tag='step_1000')
```

### Q5: 多机训练如何配置？

**使用 torchrun**：
```bash
# 节点 0（主节点）
torchrun \
    --nproc_per_node=8 \
    --nnodes=4 \
    --node_rank=0 \
    --master_addr="192.168.1.1" \
    --master_port=29500 \
    train.py

# 节点 1-3（工作节点）
torchrun \
    --nproc_per_node=8 \
    --nnodes=4 \
    --node_rank=1 \
    --master_addr="192.168.1.1" \
    --master_port=29500 \
    train.py
```

### Q6: 如何监控分布式训练？

**关键指标**：
- GPU 利用率（应 > 90%）
- 通信时间占比（应 < 20%）
- 内存使用
- 吞吐量（samples/sec）

**工具**：
- `nvidia-smi`：GPU 监控
- `nvtop`：实时 GPU 监控
- TensorBoard：训练指标
- Weights & Biases：实验跟踪

## 14. 总结与展望

### 核心要点

1. **并行策略**：
   - **数据并行**：最简单，适合中小模型
   - **模型并行**：适合超大模型
   - **流水线并行**：适合深层模型
   - **混合并行**：组合使用，训练超大模型

2. **内存优化**：
   - **ZeRO**：消除冗余，线性扩展
   - **Offload**：利用 CPU/NVMe 内存
   - **梯度检查点**：用计算换内存

3. **框架选择**：
   - **DDP**：标准数据并行
   - **FSDP**：PyTorch 原生，类似 ZeRO-3
   - **DeepSpeed**：功能最全，优化最多
   - **Accelerate**：统一接口，简化使用

4. **性能优化**：
   - 减少通信开销
   - 重叠计算和通信
   - 使用混合精度
   - 优化数据加载

### 实践建议

**从简单开始**：
1. 先用单 GPU 验证代码
2. 再用 DDP 扩展到多 GPU
3. 如果内存不够，尝试 FSDP/DeepSpeed
4. 根据需要添加更多优化

**性能调优**：
1. 测量基线性能
2. 识别瓶颈（计算 vs 通信 vs 内存）
3. 针对性优化
4. 重复迭代

### 未来趋势

1. **自动并行**：编译器自动选择并行策略
2. **异构训练**：CPU + GPU + TPU 混合
3. **弹性训练**：动态增减计算资源
4. **通信优化**：更高效的通信算法

### 下一章预告

**Module 6.3: Efficient Training Techniques**
- 混合精度训练
- 梯度检查点
- Flash Attention
- 模型编译优化

### 💡 思考题

1. 为什么 ZeRO-3 需要更多通信，但仍然值得使用？
2. 如何在流水线并行中平衡气泡时间和内存使用？
3. 什么情况下应该使用张量并行而不是流水线并行？
4. 如何设计一个训练 1T 参数模型的并行策略？

### 参考资源

**论文**：
- ZeRO: Memory Optimizations Toward Training Trillion Parameter Models
- GPipe: Easy Scaling with Micro-Batch Pipeline Parallelism
- Megatron-LM: Training Multi-Billion Parameter Language Models

**文档**：
- [PyTorch FSDP](https://pytorch.org/docs/stable/fsdp.html)
- [DeepSpeed](https://www.deepspeed.ai/)
- [Hugging Face Accelerate](https://huggingface.co/docs/accelerate/)

**工具**：
- [torchrun](https://pytorch.org/docs/stable/elastic/run.html)
- [NCCL](https://developer.nvidia.com/nccl)
- [Weights & Biases](https://wandb.ai/)

---

## 🎉 恭喜完成 Module 6.2！

你已经掌握了：
- ✅ 分布式训练的基本概念
- ✅ 数据并行、模型并行、流水线并行
- ✅ ZeRO 优化器和 DeepSpeed
- ✅ FSDP 和 Accelerate 框架
- ✅ 实践指南和性能优化

继续加油！🚀

## 思考题参考答案

### 问题 1：为什么 ZeRO-3 需要更多通信，但仍然值得使用？

**答：ZeRO-3 虽然通信量增加，但通过消除内存冗余实现了线性内存扩展，使单 GPU 无法训练的超大模型成为可能，这一价值远超通信开销。**

**通信量分析**

| 策略 | 每步通信量 | 说明 |
|------|-----------|------|
| DDP | $2\Phi$ | 1 次 AllReduce（梯度） |
| ZeRO-1 | $2\Phi$ | 1 次 AllReduce（梯度）+ 1 次 AllGather（参数） |
| ZeRO-2 | $2\Phi$ | ReduceScatter + AllGather |
| ZeRO-3 | $3\Phi$ | 前向 AllGather + 反向 AllGather + ReduceScatter |

ZeRO-3 的通信量约是 DDP 的 1.5 倍。

**为何仍值得使用**

1. **解决根本矛盾**：DDP 每个 GPU 存储完整的参数 + 梯度 + 优化器状态，对于 70B 参数模型约需 $70 \times 10^9 \times 16 \text{ bytes} \approx 1.12 \text{ TB}$，单 GPU 无论如何都无法容纳，DDP 根本无法使用

2. **线性内存扩展**：ZeRO-3 每个 GPU 内存仅需 $\frac{16\Phi}{N}$，16 个 GPU 可将内存需求降低 16 倍，使 70B 模型的每卡需求从 1.12 TB 降至 ~70 GB（A100 可容纳）

3. **通信可被重叠**：ZeRO-3 中 AllGather 操作可以与计算重叠（prefetch 下一层参数的同时计算当前层），实际通信开销远低于理论值

4. **带宽持续提升**：NVLink（600 GB/s）和 InfiniBand（200 Gb/s）的发展使通信开销不断降低，而模型规模仍在高速增长

**结论**：ZeRO-3 以约 50% 的额外通信换取了线性的内存扩展能力，这是训练千亿参数模型的必要代价。

---

### 问题 2：如何在流水线并行中平衡气泡时间和内存使用？

**答：通过调整微批次数量和调度策略，在气泡时间与激活值内存之间寻找最优平衡点。**

**气泡时间公式**

$$\text{气泡比例} = \frac{p-1}{m+p-1}$$

其中 $p$ 为流水线阶段数，$m$ 为微批次数量。

**平衡策略**

**策略 1：增加微批次数量**

增大 $m$ 可以降低气泡比例：

| 阶段数 $p$ | 微批次 $m$ | 气泡比例 |
|-----------|-----------|---------|
| 4 | 4 | 42.9% |
| 4 | 8 | 27.3% |
| 4 | 16 | 15.8% |
| 4 | 32 | 8.6% |

**代价**：增加 $m$ 意味着在前向传播阶段需要同时保存 $m$ 个微批次的中间激活值，内存压力随 $m$ 线性增长。

**策略 2：1F1B（One-Forward-One-Backward）调度**

Pipedream-Flush 等调度算法交替进行前向和反向传播：

```
GPU 0: F0  F1  F2  F3  B3  B2  B1  B0
GPU 1:  F0  F1  F2  B2  B1  B0
GPU 2:   F0  F1  B1  B0
GPU 3:    F0  B0
```

1F1B 调度的最大内存占用不随 $m$ 增长，而是固定为 $p$ 个微批次的激活值。

**策略 3：梯度检查点配合使用**

在每个流水线阶段内使用梯度检查点，进一步降低激活值内存，允许使用更大的 $m$ 以减少气泡。

**实践建议**

- 一般选择 $m \geq 2p$（使气泡比例低于 33%）
- 内存受限时优先采用 1F1B 调度
- 结合梯度检查点和混合精度进一步节省内存

---

### 问题 3：什么情况下应该使用张量并行而不是流水线并行？

**答：两种并行方式针对不同瓶颈，应根据模型结构、单层大小和 GPU 间互联带宽来选择。**

**核心区别**

| 维度 | 张量并行 | 流水线并行 |
|------|---------|----------|
| 切分方式 | 按参数矩阵维度切分单个层 | 按层切分整个模型 |
| 通信频率 | 每个 Transformer 子层 2 次 AllReduce | 只在阶段边界传递激活值 |
| 通信量 | 激活值大小（$B \times L \times H$） | 激活值大小（$B \times L \times H$）|
| 气泡时间 | 无气泡（同步并行） | 有气泡（需微批次缓解） |
| 适用层 | 单层参数极大 | 层数极多 |

**何时选择张量并行**

1. **单层参数矩阵极大**：例如 FFN 层宽度为 $d_{\text{model}} = 8192$，单个权重矩阵 $W \in \mathbb{R}^{8192 \times 32768}$（~1GB），单 GPU 无法容纳
2. **GPU 间互联带宽极高**：张量并行每前向一层需要 AllReduce，要求节点内 NVLink 级别带宽（>100 GB/s）；跨节点带宽不足时张量并行效率急剧下降
3. **模型层数不多但层很宽**：流水线并行要求有足够多的层可供切分

**何时选择流水线并行**

1. **模型层数很多（>24 层）**：每个 GPU 负责若干完整的层，层间通信只在边界发生
2. **跨节点训练**：流水线并行的通信量与激活值大小成正比，通常远小于张量并行的通信频率，适合带宽较低的跨节点场景
3. **可以忍受气泡时间**：通过增加微批次数量将气泡比例控制在 10% 以内

**最佳实践：混合并行**

在实际超大模型训练中（如 Megatron-LM 训练 GPT-3 系列），通常采用三维混合并行：

- **节点内**：张量并行（高带宽 NVLink）
- **节点间**：流水线并行（InfiniBand）
- **多副本**：数据并行（ZeRO 内存优化）

---

### 问题 4：如何设计一个训练 1T（万亿）参数模型的并行策略？

**答：需要将数据并行、张量并行、流水线并行三者组合，并配合 ZeRO 内存优化，同时兼顾硬件拓扑结构。**

**资源估算**

1T 参数模型（以 BF16 存储参数，FP32 优化器状态）所需内存：

$$M_{\text{per GPU}} = \frac{16 \times 10^{12} \text{ bytes}}{N_{\text{GPU}}} = \frac{16 \text{ TB}}{N_{\text{GPU}}}$$

使用 A100 80GB GPU 需要至少 $\lceil 16 \text{ TB} / 80 \text{ GB} \rceil = 200$ 个 GPU 仅用于存储参数（不含激活值）。

**三维并行设计**

假设使用 1024 个 A100（每节点 8 卡）：

| 并行维度 | 并行度 | 说明 |
|---------|--------|------|
| 张量并行 $t$ | 8 | 节点内 NVLink，每节点 8 卡同并行 |
| 流水线并行 $p$ | 16 | 跨 16 个节点，每个阶段 8 卡 |
| 数据并行 $d$ | 8 | 8 个完整模型副本（ZeRO-1 优化器分片）|

总 GPU 数：$t \times p \times d = 8 \times 16 \times 8 = 1024$

**模型切分方案（以 Transformer 为例）**

假设模型有 128 层，$d_{\text{model}} = 12288$，$d_{\text{ff}} = 49152$：

- **张量并行**：将每层 Attention 的 Q/K/V 按列切分，FFN 第一层列切分、第二层行切分，每个 GPU 负责 $d_{\text{model}} / 8 = 1536$ 列
- **流水线并行**：128 层分为 16 个阶段，每个阶段 8 层
- **ZeRO-1**：8 个数据并行副本之间分片优化器状态，节省 8× 优化器内存

**关键工程挑战**

1. **负载均衡**：流水线各阶段计算量相等，Embedding 层和最后的 LM Head 通常单独放一个阶段
2. **通信拓扑感知**：张量并行放节点内（高带宽），流水线并行跨节点（低延迟）
3. **检查点策略**：梯度检查点 + 1F1B 调度控制激活值内存
4. **容错机制**：1024 GPU 故障概率极高，需要弹性训练和周期性保存检查点

**参考实现**：Megatron-LM、DeepSpeed-Megatron 实现了类似设计，用于训练 Megatron-Turing NLG（530B）和 GPT-NeoX（20B）等超大模型。
